**1.1 Install dependencies**

In [ ]:
!pip install -U datasets
!pip install transformers  -q

**2. Practical Case**

In [ ]:
# ✅ Paso 1: Install Required Libraries
!pip install -q pymupdf transformers
import fitz  # PyMuPDF
from google.colab import files

from transformers import pipeline
import math
import re
import unicodedata
import pandas as pd
from IPython.display import Markdown, display


In [ ]:
def clean_text(text):
    # Reemplaza caracteres invisibles o especiales
    text = text.replace("\xa0", " ")  # NBSP a espacio normal
    text = unicodedata.normalize("NFKC", text)  # Normaliza caracteres Unicode

    # Limpieza de espacios y saltos de línea
    text = re.sub(r"[ \t]+", " ", text)              # Múltiples espacios/tab por uno
    text = re.sub(r"\s*\n\s*", "\n", text)           # Saltos de línea limpios
    text = re.sub(r"\n{2,}", "\n", text)             # Evita dobles saltos

    # Elimina espacios antes de puntuación
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)

    # Corrige espacios duplicados finales
    return text.strip()

In [ ]:
# ✅ Paso 2: Cargar el contrato en pdf

uploaded = files.upload()  # Se debe seleccionar el documento en pdf

# ✅ Paso 3: Extraer el texto de PDF usando PyMupdf


filename = next(iter(uploaded))  # Automatizar la subida del archivo
doc = fitz.open(filename)

# Extraer el texto
full_text = ""
for page in doc:
    full_text += page.get_text()

print(f"✅ Extracted {len(full_text)} characters from PDF")

# ✅ Paso 4: Cargar el Summarization Model de Hugging Face



summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# ✅ Paso 5: Break long text into manageable chunks (BART max length ~1024 tokens)


def chunk_text(text, max_chunk_length=1024):
    paragraphs = text.split("\n")
    chunks = []
    current = ""

    for para in paragraphs:
        if len(current) + len(para) <= max_chunk_length:
            current += para + "\n"
        else:
            chunks.append(current)
            current = para + "\n"
    if current:
        chunks.append(current)
    return chunks

chunks = chunk_text(full_text, max_chunk_length=1024)

In [ ]:
# ✅ Paso 6: Generar Summaries for Each Chunk
print(f"📚 Summarizing {len(chunks)} chunks...")
intermediate_summaries = []

for chunk in chunks:
    summary = summarizer(chunk, max_length=150, min_length=40, do_sample=False)[0]['summary_text']
    intermediate_summaries.append(summary)



In [ ]:
# ✅ Paso 7: Limpiar los mini-summaries antes del resumen final
cleaned_summaries = [clean_text(s) for s in intermediate_summaries]
combined = " ".join(cleaned_summaries)


executive_summary = summarizer(
    combined,
    max_length=600,
    min_length=400,
    do_sample=False
)[0]["summary_text"]

In [ ]:
executive_summary

In [ ]:
display(Markdown(f"### 📄 RESUMEN EJECUTIVO\n\n{executive_summary}"))

**2.1. Otros modelos - Q&A**

> Con el afán de entender mejor estos textos, podemos introducir más modelos que nos permitan obtener insights de los textos. Por ejemplo, utilizaremos el modelo **mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es** para hacer preguntas y respuestas de cosas específicas en el texto



In [ ]:
qa_es = pipeline("question-answering", model="mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es")

question = "¿Cuál es la duración del contrato?"
answer = qa_es(question=question, context=full_text)

print(f"Q: {question}\nA: {answer['answer']} (score: {answer['score']:.2f})")

In [ ]:
qa_es = pipeline("question-answering", model="mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es")

question = "¿Cuál es la dirección del inmueble?"
answer = qa_es(question=question, context=full_text)

print(f"Q: {question}\nA: {answer['answer']} (score: {answer['score']:.2f})")

In [ ]:
qa_es = pipeline("question-answering", model="mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es")

question = "¿Cuál es el codigo interbancario de la arrendadora"
answer = qa_es(question=question, context=full_text)

print(f"Q: {question}\nA: {answer['answer']} (score: {answer['score']:.2f})")

**2.2. Otros modelos - Clasificador de cláusulas**

 Se agregó un modelo de clasificación de cláusulas legales de manera que una persona en vez de leer todo el texto podría irse de frente a la cláusula que le llama la atención según la clasificación que obtenga.    Este caso lo veo más aplicable para derecho. El modelo usado es **joeddav/xlm-roberta-large-xnli**

In [ ]:
classifier = pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli")

# Divide usando líneas con todo mayúsculas que son típicas de encabezados legales
clausulas = re.split(r'\n(?=\s*[A-ZÑÁÉÍÓÚÜ]{3,}[^\n]*:)', full_text)

# Filtrar si hay cláusulas muy cortas o vacías
clausulas = [c.strip() for c in clausulas if len(c.strip()) > 50]

# ✅ Paso 4: Clasificar cada cláusula con Zero-Shot
resultados = []



labels = ["Pago", "Confidencialidad", "Terminación", "Obligaciones", "Jurisdicción"]

for idx, clausula in enumerate(clausulas):
    result = classifier(clausula[:1000], candidate_labels=labels)
    resultados.append({
        "cláusula": clausula.split("\n")[0].strip(),  # Título o primera línea
        "más probable": result["labels"][0],
        "score": result["scores"][0],
        "Scores": list(zip(result["labels"], result["scores"]))
    })

# ✅ Paso 5: Mostrar en tabla (puedes exportar si quieres)
import pandas as pd

df = pd.DataFrame([{
    "Cláusula": r["cláusula"],
    "Categoría detectada": r["más probable"],
    "Puntaje": round(r["score"], 2),
    "Etiquetas": ", ".join([f"{l} ({s:.2f})" for l, s in r["Scores"]])
} for r in resultados])

pd.set_option('display.max_colwidth', None)
display(df)

**2.3. Otros modelos - Clasificador de cláusulas**

 Se agregó un modelo de clasificación de cláusulas legales de manera que una persona en vez de leer todo el texto podría irse de frente a la cláusula que le llama la atención según la clasificación que obtenga.    Este caso lo veo más aplicable para derecho. El modelo usado es **joeddav/xlm-roberta-large-xnli**

In [ ]:


# Inicializar el pipeline
ner_es = pipeline("ner", model="mrm8488/bert-spanish-cased-finetuned-ner", grouped_entities=True)

# Cortar texto seguro
chunks = textwrap.wrap(full_text, width=400)
all_entities = []

for chunk in chunks:
    try:
        ents = ner_es(chunk)
        all_entities.extend(ents)
    except Exception as e:
        print(f"❌ Error en chunk: {e}")

# Limpiar entidades
def clean_entity(e):
    word = re.sub(r"##", "", e["word"]).strip()
    word = word.strip(",.():;–-")
    return {
        "Entidad detectada": word,
        "Tipo": e["entity_group"],
        "Confianza": round(e["score"], 2)
    }

# Aplicar limpieza
ent_clean = [clean_entity(e) for e in all_entities if len(e["word"]) > 2 and not e["word"].startswith("##")]

# Quitar duplicados por entidad + tipo
df_entities = pd.DataFrame(ent_clean).drop_duplicates(subset=["Entidad detectada", "Tipo"])



In [ ]:
# Ordenar por confianza descendente
df_entities = df_entities.sort_values(by="Confianza", ascending=False).reset_index(drop=True)

# Mostrar tabla final ordenada
pd.set_option('display.max_colwidth', None)
display(df_entities)